# Mind Quest — Group 16 (PIPS)

Mind Quest is an educational jump-and-run game about general psychology, written in Python with pygame.
A prologue introduces Open Science, and three levels teach **attention**, **response inhibition** and **executive control**.
Each level contains a real experiment whose data are saved as CSV files in `data/`:

| Level | Experiment | Data file |
| --- | --- | --- |
| Level 1: Focus Fields | child flanker task (Rueda et al., 2004) | `flanker_*.csv` |
| Level 2: Hold Your Horses | stop-signal task (Verbruggen et al., 2019) | `stopsignal_*.csv` |
| Level 3: Mission Control | 2-back task (Kirchner, 1958), Tower of London (Shallice, 1982) | `nback_*.csv`, `tower_*.csv` |
| Level 3 boss | card sorting with the Rule Keeper (Milner, 1963) | `cardsort_*.csv` |

This notebook

1. starts the game (part 1), and
2. loads and summarises the data written by the experiments (part 2).

See `README.md` for the controls and a full description of the game.

## Setup

The game needs Python 3.12 or newer and the packages in `requirements.txt`. Uncomment the line below to install them from the notebook.

In [ ]:
# %pip install -r requirements.txt

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

DATA_DIR = Path("data")

## Part 1: Play the game

Running the cell below opens the game window. Play as long as you like; close the window (or quit from the menu) to return to the notebook.

* `participant_id` is an anonymous label used in the data files. Leave it as `None` to get a random label such as `p-3fa2c1`.
* `allow_admin=False` switches off the admin mode (P / F1), so challenges cannot be skipped. Use this when you collect data.
* `seed` makes the trial orders reproducible.

Set `PLAY_GAME = False` to skip the game and only analyse data that are already in `data/`.

In [ ]:
PLAY_GAME = True

if PLAY_GAME:
    from src.game import run_game

    saved_files = run_game(participant_id=None, allow_admin=False)
    print(f"{len(saved_files)} data file(s) saved:")
    for path in saved_files:
        print(" ", path)

## Part 2: The experiment data

Every attempt at an experiment writes one file `data/<task>_<participant>_<timestamp>_attempt-<n>.csv` with one row per trial.
The helper below reads all files of one task into a single table. Only the **test** block is analysed; practice trials are left out.

In [ ]:
def load_task(task, data_dir=DATA_DIR):
    """Read all CSV files of one task into one DataFrame (empty if there are none)."""
    files = sorted(Path(data_dir).glob(f"{task}_*.csv"))
    if not files:
        print(f"No {task} data found in {data_dir}/")
        return pd.DataFrame()
    data = pd.concat([pd.read_csv(f).assign(file=f.name) for f in files], ignore_index=True)
    print(f"{task}: {len(files)} file(s), {len(data)} rows, participants: {', '.join(data['participant_id'].unique())}")
    return data


def test_block(data):
    """Keep only the test trials if the data have a block column."""
    if "block" in data.columns:
        return data[data["block"] == "test"].copy()
    return data

### Level 1: Flanker task

Five fish are shown and the player indicates the direction of the **middle** fish. The flankers swim the same way (*congruent*), the other way (*incongruent*), or are absent (*neutral*).
The **flanker effect** is the mean correct reaction time on incongruent minus congruent trials (Eriksen & Eriksen, 1974). Trials without a response count as errors.

In [ ]:
flanker = test_block(load_task("flanker"))

if not flanker.empty:
    conditions = ["congruent", "neutral", "incongruent"]
    accuracy = flanker.pivot_table(index=["participant_id", "attempt"], columns="condition",
                                   values="correct", aggfunc="mean").reindex(columns=conditions)
    mean_rt = (flanker[flanker["correct"]]
               .pivot_table(index=["participant_id", "attempt"], columns="condition",
                            values="rt_ms", aggfunc="mean")
               .reindex(columns=conditions))
    summary = mean_rt.add_suffix("_rt_ms").round(0)
    summary["accuracy"] = flanker.groupby(["participant_id", "attempt"])["correct"].mean().round(2)
    summary["flanker_effect_ms"] = (mean_rt["incongruent"] - mean_rt["congruent"]).round(0)
    display(summary)

In [ ]:
if not flanker.empty:
    correct = flanker[flanker["correct"]]
    rt = correct.groupby("condition")["rt_ms"].agg(["mean", "sem"]).reindex(conditions)
    acc = flanker.groupby("condition")["correct"].mean().reindex(conditions)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].bar(rt.index, rt["mean"], yerr=rt["sem"].fillna(0), capsize=4, color=["#4c9f70", "#9aa5b1", "#d1603d"])
    axes[0].set(title="Mean correct RT per condition", ylabel="Reaction time (ms)")
    axes[1].bar(acc.index, acc * 100, color=["#4c9f70", "#9aa5b1", "#d1603d"])
    axes[1].set(title="Accuracy per condition", ylabel="Correct (%)", ylim=(0, 100))
    fig.suptitle("Flanker task (test trials, all attempts)")
    fig.tight_layout()
    plt.show()

### Level 2: Stop-signal task

A race car points left or right (A / D). On a quarter of the trials a STOP sign follows after the stop-signal delay (SSD), and no key should be pressed. The SSD follows a staircase in steps of 50 ms.
The **stop-signal reaction time (SSRT)** is estimated with the integration method (Verbruggen et al., 2019): missed go trials get the longest go RT, the go RTs are sorted, and the RT at the rank that matches the probability of responding on stop trials minus the mean SSD is the SSRT.

In [ ]:
import math


def ssrt_integration(go, stop):
    """SSRT with the integration method, as in src/game/stop_signal_task.py."""
    if go.empty or stop.empty:
        return float("nan")
    longest = go["rt_ms"].max()
    go_rts = sorted(go["rt_ms"].fillna(longest))
    p_respond = 1 - stop["correct"].mean()
    rank = min(len(go_rts), max(1, math.ceil(p_respond * len(go_rts))))
    return go_rts[rank - 1] - stop["ssd_ms"].mean()


stop_signal = test_block(load_task("stopsignal"))

if not stop_signal.empty:
    rows = []
    for (pid, attempt), trials in stop_signal.groupby(["participant_id", "attempt"]):
        go, stop = trials[trials["trial_type"] == "go"], trials[trials["trial_type"] == "stop"]
        rows.append({
            "participant_id": pid,
            "attempt": attempt,
            "go_accuracy": go["correct"].mean(),
            "mean_go_rt_ms": go.loc[go["correct"], "rt_ms"].mean(),
            "stop_success": stop["correct"].mean(),
            "mean_ssd_ms": stop["ssd_ms"].mean(),
            "ssrt_ms": ssrt_integration(go, stop),
        })
    display(pd.DataFrame(rows).set_index(["participant_id", "attempt"]).round(2))

In [ ]:
if not stop_signal.empty:
    fig, ax = plt.subplots(figsize=(8, 4))
    for (pid, attempt), trials in stop_signal.groupby(["participant_id", "attempt"]):
        stops = trials[trials["trial_type"] == "stop"]
        ax.plot(range(1, len(stops) + 1), stops["ssd_ms"], marker="o", label=f"{pid}, attempt {attempt}")
    ax.set(title="Stop-signal delay staircase", xlabel="Stop trial", ylabel="SSD (ms)")
    ax.legend()
    plt.show()

### Level 3: 2-back task

Letters appear one by one, and the player presses SPACE when a letter equals the one two steps back. Sensitivity is summarised with **d′** using the log-linear correction (Hautus, 1995).

In [ ]:
from statistics import NormalDist


def d_prime(hits, misses, false_alarms, correct_rejections):
    """d' with the log-linear correction (Hautus, 1995), as in src/game/nback_task.py."""
    hit_rate = (hits + 0.5) / (hits + misses + 1)
    false_alarm_rate = (false_alarms + 0.5) / (false_alarms + correct_rejections + 1)
    return NormalDist().inv_cdf(hit_rate) - NormalDist().inv_cdf(false_alarm_rate)


nback = test_block(load_task("nback"))

if not nback.empty:
    rows = []
    for (pid, attempt), trials in nback.groupby(["participant_id", "attempt"]):
        target, responded = trials["is_target"], trials["responded"]
        hits, misses = (target & responded).sum(), (target & ~responded).sum()
        fas, crs = (~target & responded).sum(), (~target & ~responded).sum()
        rows.append({
            "participant_id": pid,
            "attempt": attempt,
            "accuracy": trials["correct"].mean(),
            "hit_rate": hits / (hits + misses) if hits + misses else float("nan"),
            "false_alarm_rate": fas / (fas + crs) if fas + crs else float("nan"),
            "d_prime": d_prime(hits, misses, fas, crs),
            "mean_hit_rt_ms": trials.loc[target & responded, "rt_ms"].mean(),
        })
    display(pd.DataFrame(rows).set_index(["participant_id", "attempt"]).round(2))

### Level 3: Tower of London

One row per puzzle. Planning time is the time until the first ball is lifted; *extra moves* is the number of moves above the shortest solution.

In [ ]:
tower = load_task("tower")

if not tower.empty:
    tower["extra_moves"] = tower["moves"] - tower["min_moves"]
    display(tower.groupby("min_moves")[["solved", "extra_moves", "planning_time_ms", "solution_time_ms"]].mean().round(2))

    fig, ax = plt.subplots(figsize=(6, 4))
    planning = tower.groupby("min_moves")["planning_time_ms"].mean()
    ax.bar(planning.index.astype(str), planning / 1000, color="#5b7db1")
    ax.set(title="Tower of London: planning time", xlabel="Minimum number of moves", ylabel="Planning time (s)")
    plt.show()

### Level 3 boss: Card sorting with the Rule Keeper

The hidden sorting rule (colour, shape or number) changes without warning. A **perseverative error** is a wrong choice that follows the previous rule (Milner, 1963).

In [ ]:
cardsort = load_task("cardsort")

if not cardsort.empty:
    display(cardsort.groupby(["participant_id", "attempt"]).agg(
        n_cards=("trial", "size"),
        accuracy=("correct", "mean"),
        perseverative_errors=("perseverative", "sum"),
        rule_changes=("rule", lambda rules: int((rules != rules.shift()).sum() - 1)),
    ).round(2))

## References

The full list of sources is shown in the game (home screen, "References") and is available in code:

In [ ]:
from src.game.content import REFERENCES

for reference in REFERENCES:
    print(reference, end="\n\n")